# Synthefy's 2026 fantasy football rankings with Nori

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Synthefy/synthefy-nori/blob/main/examples/notebooks/nori-fantasy-football-2026.ipynb)

This notebook builds the fantasy-football forecast behind the Synthefy blog post from public sources. It:

1. creates leak-safe player-season rows from [nflverse](https://github.com/nflverse) and the free [Fantasy Football Calculator API](https://fantasyfootballcalculator.com/api-docs);
2. evaluates Nori-6M on a 2025 holdout using only 2015–2024 context;
3. forecasts low, expected, and high first-15-game PPR totals for 2026; and
4. publishes the top 200 strictly by expected points.

No dataset or checkpoint is bundled in the repository. The notebook downloads public inputs at runtime and records their hashes.


## Setup

For reproduction, the notebook pins the package used for the published run and verifies the public Nori-6M checkpoint hash. A Colab GPU is recommended; `fit()` only stores the context rows, while `predict()` performs inference.


In [ ]:
import importlib.metadata
import subprocess
import sys

REQUIRED_PACKAGES = {
    "synthefy-nori": "0.20.0",
    "nflreadpy": "0.1.5",
}

needs_install = any(
    importlib.metadata.version(package) != wanted
    if package in {dist.metadata["Name"] for dist in importlib.metadata.distributions()}
    else True
    for package, wanted in REQUIRED_PACKAGES.items()
)
if needs_install:
    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "synthefy-nori==0.20.0",
            "nflreadpy==0.1.5",
        ]
    )


In [ ]:
import hashlib
import json
import re
import unicodedata
import urllib.request
import warnings
from collections import defaultdict
from difflib import SequenceMatcher

import nflreadpy as nfl
import numpy as np
import pandas as pd
from huggingface_hub import hf_hub_download
from sklearn.metrics import mean_absolute_error
from synthefy_nori import NoriRegressor

START_SEASON = 2015
HOLDOUT_SEASON = 2025
FORECAST_SEASON = 2026
TARGET_THROUGH_WEEK = 16
TARGET_GAMES = 15
N_LAGS = 5
ZERO_FILL_LAG_HISTORY = True
POSITIONS = {"QB", "RB", "WR", "TE"}
POSITION_CODE = {"QB": 0.0, "RB": 1.0, "WR": 2.0, "TE": 3.0}
MODEL = "nori-6m"

# FFC responses are mutable. These canonical JSON hashes identify the September 2 run.
EXPECTED_FFC_SHA256 = {
    2015: "18345becfb813d5b7c8ecaa09159d7f3028941fbb355b183ce4f5499d4e3ad20",
    2016: "eafc5a14ece4845bbc57e4522daf237f0186159c9fc80b100b5b6052cd9fb090",
    2017: "56caa8abcc4aafe9e7e544f049e0e5ad8cd25bec3aaec2301bcddc93ef506f04",
    2018: "deca10faf3ccfe821f9ac8391c7412f8986ba9e55d2f4b75e2cdcb5ef72d6cb6",
    2019: "b481d4c687c1c7b357e12065c9b6c22f3d1c2542434b18d41167b65305e6e883",
    2020: "59f44898b078d34b7a1dbf0638ef9bd97b4ee39f18d2be06891d32f3c2e8a33d",
    2021: "e4e34fa05ecf61d865753c695d7ffd6ab0adc60019a92e9142a2a572e9058451",
    2022: "6bf1cb8b168b287db32a7f9509d4334eb3a484d1224c871830fee16588e6e1bc",
    2023: "67ac90921bf68e036bcd6c670a20c99e988c872f395cd46f09a65efa3be5078e",
    2024: "c43ad1aa3f5cb3a11df3250182f61561bf13b324224d4d787d90d5810e41bc7e",
    2025: "73dc3a45f98b7bbb20ddfe29fb135be8154fcd2dc5f371c181d934cb1491fc2e",
    2026: "c7101f7c894fde3f8b599556c50fed387ca5810e03d86cdb7b983838e8254e14",
}
EXPECTED_NORI_6M_SHA256 = "a13b2bc31d8db24d17bae6d04844e0adf669e446087b0b7a34c7b05045d61323"

# Exact score from the normalized five-lag reproduction below.
EXPECTED_2025_MAE = 51.48745485472745
EXPECTED_FEATURE_TABLE_SHA256 = "f07b1b6069d4faa19f8993df4f8e6b5d0f9abf358769604a8d0e68b345ac2d79"


## Build one row per player and preseason

The target is PPR points through NFL Week 16, when every team has played 15 games. A row for season *t* uses preseason average draft position, player metadata known before Week 1, and production from seasons *t−1* through *t−5*. Unavailable numeric lag history is zero-filled, so every player has the same five-season input window.

The 2025 holdout receives no 2025 statistics as features. The 2026 query rows receive 2025 and earlier statistics, including Matthew Stafford's 2025 fantasy points and games played.


In [ ]:
STAT_COLUMNS = [
    "games",
    "fantasy_points_ppr",
    "attempts",
    "passing_yards",
    "passing_tds",
    "passing_interceptions",
    "carries",
    "rushing_yards",
    "rushing_tds",
    "targets",
    "receptions",
    "receiving_yards",
    "receiving_tds",
    "target_share",
    "air_yards_share",
    "wopr",
]
ALIASES = {
    "hollywoodbrown": "marquisebrown",
    "gabedavis": "gabrieldavis",
    "chigokonkwo": "chigoziemokonkwo",
    "mitchtrubisky": "mitchelltrubisky",
    "willfuller": "willfullerv",
    "kennygainwell": "kennethgainwell",
}
TEAM_ALIASES = {"LA": "LAR"}


def sha256_bytes(value: bytes) -> str:
    return hashlib.sha256(value).hexdigest()


def sha256_file(path) -> str:
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def normalize_team(value):
    return TEAM_ALIASES.get(value, value) if isinstance(value, str) else value


def normalize_name(value) -> str:
    if value is None or pd.isna(value):
        return ""
    text = unicodedata.normalize("NFKD", str(value)).encode("ascii", "ignore").decode().lower()
    text = re.sub(r"\b(jr|sr|ii|iii|iv|v)\b", "", text)
    text = re.sub(r"[^a-z0-9]", "", text)
    return ALIASES.get(text, text)


def numeric(value) -> float:
    try:
        return float(value)
    except (TypeError, ValueError):
        return np.nan


def age_on_september_1(birth_date, season: int) -> float:
    date = pd.to_datetime(birth_date, errors="coerce")
    if pd.isna(date):
        return np.nan
    start = pd.Timestamp(year=season, month=9, day=1)
    return (start - date).days / 365.25


def fetch_ffc(season: int) -> tuple[pd.DataFrame, dict]:
    url = f"https://fantasyfootballcalculator.com/api/v1/adp/ppr?teams=12&year={season}"
    request = urllib.request.Request(url, headers={"User-Agent": "synthefy-nori-notebook/1.0"})
    with urllib.request.urlopen(request, timeout=60) as response:
        body = response.read()
    payload = json.loads(body)
    canonical = json.dumps(payload, sort_keys=True, separators=(",", ":")).encode()
    rows = pd.DataFrame(payload["players"])
    rows["season"] = season
    rows["position"] = rows["position"].replace({"PK": "K"})
    audit = {
        "season": season,
        "url": url,
        "rows": len(rows),
        "sha256": sha256_bytes(canonical),
    }
    return rows.loc[rows["position"].isin(POSITIONS)].copy(), audit


def build_alias_index(players: pd.DataFrame):
    by_exact = defaultdict(set)
    by_name = defaultdict(set)
    by_position = defaultdict(list)
    for row in players.itertuples(index=False):
        if not isinstance(row.gsis_id, str):
            continue
        aliases = {
            normalize_name(getattr(row, column, None))
            for column in ("display_name", "football_name")
        }
        first = getattr(row, "common_first_name", None) or getattr(row, "first_name", None)
        aliases.add(normalize_name(f"{first or ''} {getattr(row, 'last_name', '') or ''}"))
        for alias in aliases - {""}:
            by_exact[(row.position, alias)].add(row.gsis_id)
            by_name[alias].add(row.gsis_id)
            if row.position in POSITIONS:
                by_position[row.position].append((alias, row.gsis_id))
    return by_exact, by_name, by_position


def resolve_candidates(ids, season, team, position, players, stats_index):
    scored = []
    for player_id in ids:
        if player_id not in players.index:
            continue
        metadata = players.loc[player_id]
        rookie = numeric(metadata.get("rookie_season"))
        if not np.isfinite(rookie):
            rookie = numeric(metadata.get("draft_year"))
        last = numeric(metadata.get("last_season"))
        score = 10.0 if np.isfinite(rookie) and rookie <= season and (
            not np.isfinite(last) or season <= last
        ) else 0.0
        score += float(metadata.get("position") == position)
        if (season, player_id) in stats_index.index:
            score += 5.0
            if normalize_team(stats_index.loc[(season, player_id)].get("recent_team")) == normalize_team(team):
                score += 3.0
        score += 2.0 * float((season - 1, player_id) in stats_index.index)
        scored.append((score, player_id))
    scored.sort(reverse=True)
    if scored and (len(scored) == 1 or scored[0][0] > scored[1][0]):
        return scored[0][1]
    return None


def match_player(name, position, season, team, indexes, players, stats_index):
    by_exact, by_name, by_position = indexes
    normalized = normalize_name(name)
    for candidates, method in (
        (by_exact.get((position, normalized), set()), "exact"),
        (by_name.get(normalized, set()), "exact_cross_position"),
    ):
        resolved = resolve_candidates(candidates, season, team, position, players, stats_index)
        if resolved is not None:
            return resolved, method
    similarity_by_id = {}
    for alias, player_id in by_position[position]:
        score = SequenceMatcher(None, normalized, alias).ratio()
        similarity_by_id[player_id] = max(similarity_by_id.get(player_id, 0.0), score)
    ordered = sorted(similarity_by_id.items(), key=lambda item: item[1], reverse=True)
    if ordered and ordered[0][1] >= 0.90 and (
        len(ordered) == 1 or ordered[0][1] - ordered[1][1] >= 0.025
    ):
        return ordered[0][0], "fuzzy"
    return None, "unmatched"


In [ ]:
def first_15_game_targets(seasons: list[int]) -> pd.DataFrame:
    schedules = nfl.load_schedules(seasons).to_pandas()
    regular = schedules[
        (schedules["game_type"] == "REG")
        & (schedules["week"] <= TARGET_THROUGH_WEEK)
    ]
    team_games = pd.concat(
        [
            regular[["season", "game_id", "home_team"]].rename(columns={"home_team": "team"}),
            regular[["season", "game_id", "away_team"]].rename(columns={"away_team": "team"}),
        ],
        ignore_index=True,
    )
    counts = team_games.groupby(["season", "team"])["game_id"].nunique()
    if not (counts == TARGET_GAMES).all():
        raise ValueError(f"Week 16 is not a 15-team-game horizon:\n{counts[counts != TARGET_GAMES]}")

    weekly = nfl.load_player_stats(seasons, summary_level="week").to_pandas()
    weekly = weekly[
        (weekly["season_type"] == "REG")
        & (weekly["week"] <= TARGET_THROUGH_WEEK)
        & weekly["position"].isin(POSITIONS)
    ]
    return (
        weekly.groupby(["season", "player_id"], as_index=False)
        .agg(actual_ppr=("fantasy_points_ppr", "sum"), actual_games=("game_id", "nunique"))
    )


def build_player_seasons():
    stat_seasons = list(range(START_SEASON - N_LAGS, FORECAST_SEASON))
    summary = nfl.load_player_stats(stat_seasons, summary_level="reg").to_pandas()
    summary = summary.loc[summary["position"].isin(POSITIONS)]
    summary = summary.sort_values(["season", "player_id"]).drop_duplicates(
        ["season", "player_id"], keep="last"
    )
    stats_index = summary.set_index(["season", "player_id"])

    player_metadata = nfl.load_players().to_pandas()
    player_metadata = player_metadata.drop_duplicates("gsis_id", keep="last")
    players = player_metadata.set_index("gsis_id", drop=False)
    indexes = build_alias_index(player_metadata)

    targets = first_15_game_targets(list(range(START_SEASON, HOLDOUT_SEASON + 1)))
    target_index = targets.set_index(["season", "player_id"])

    rows = []
    source_audit = []
    match_audit = []
    for season in range(START_SEASON, FORECAST_SEASON + 1):
        adp, audit = fetch_ffc(season)
        source_audit.append(audit)
        for player in adp.to_dict(orient="records"):
            player_id, method = match_player(
                player["name"],
                player["position"],
                season,
                player.get("team"),
                indexes,
                players,
                stats_index,
            )
            match_audit.append(
                {
                    "season": season,
                    "name": player["name"],
                    "position": player["position"],
                    "method": method,
                }
            )
            if player_id is None:
                continue

            metadata = players.loc[player_id]
            draft_year = numeric(metadata.get("draft_year"))
            rookie_season = numeric(metadata.get("rookie_season"))
            career_start = rookie_season if np.isfinite(rookie_season) else draft_year
            row = {
                "season": season,
                "player_id": player_id,
                "player_name": player["name"],
                "position": player["position"],
                "team": normalize_team(player.get("team")),
                "adp": numeric(player.get("adp")),
                "adp_sd": numeric(player.get("stdev")),
                "times_drafted": numeric(player.get("times_drafted")),
                "adp_high": numeric(player.get("high")),
                "adp_low": numeric(player.get("low")),
                "age": age_on_september_1(metadata.get("birth_date"), season),
                "rookie": float(draft_year == season),
                "years_since_draft": max(0.0, season - draft_year)
                if np.isfinite(draft_year)
                else np.nan,
                "pro_experience": max(0.0, season - career_start)
                if np.isfinite(career_start)
                else np.nan,
                "draft_round": numeric(metadata.get("draft_round")),
                "draft_pick": numeric(metadata.get("draft_pick")),
                "height": numeric(metadata.get("height")),
                "weight": numeric(metadata.get("weight")),
                "match_method": method,
            }

            target_key = (season, player_id)
            if season <= HOLDOUT_SEASON and target_key in target_index.index:
                target = target_index.loc[target_key]
                row["actual_ppr"] = numeric(target["actual_ppr"])
                row["actual_games"] = numeric(target["actual_games"])
            elif season <= HOLDOUT_SEASON:
                row["actual_ppr"] = 0.0
                row["actual_games"] = 0.0
            else:
                row["actual_ppr"] = np.nan
                row["actual_games"] = np.nan

            for lag in range(1, N_LAGS + 1):
                previous = (
                    stats_index.loc[(season - lag, player_id)]
                    if (season - lag, player_id) in stats_index.index
                    else None
                )
                for column in STAT_COLUMNS:
                    value = numeric(previous[column]) if previous is not None else np.nan
                    if ZERO_FILL_LAG_HISTORY and not np.isfinite(value):
                        value = 0.0
                    row[f"prev{lag}_{column}"] = value
                games = row[f"prev{lag}_games"]
                points = row[f"prev{lag}_fantasy_points_ppr"]
                row[f"prev{lag}_ppg"] = points / games if games > 0 else 0.0
                row[f"prev{lag}_team"] = (
                    normalize_team(previous["recent_team"]) if previous is not None else None
                )

            row["changed_team"] = float(
                isinstance(row["prev1_team"], str)
                and isinstance(row["team"], str)
                and row["prev1_team"] != row["team"]
            )
            row["points_trend"] = (
                row["prev1_fantasy_points_ppr"] - row["prev2_fantasy_points_ppr"]
            )
            row["ppg_trend"] = row["prev1_ppg"] - row["prev2_ppg"]
            rows.append(row)

    frame = pd.DataFrame(rows).sort_values(["season", "adp"]).reset_index(drop=True)
    return frame, pd.DataFrame(source_audit), pd.DataFrame(match_audit), summary


frame, source_audit, match_audit, season_stats = build_player_seasons()

source_audit["expected_sha256"] = source_audit["season"].map(EXPECTED_FFC_SHA256)
source_audit["matches_frozen_source"] = (
    source_audit["sha256"] == source_audit["expected_sha256"]
)
if not source_audit["matches_frozen_source"].all():
    warnings.warn(
        "At least one live FFC response differs from the frozen source. "
        "The holdout metric or 2026 board may therefore differ."
    )

assert frame.loc[frame["season"] == FORECAST_SEASON, "actual_ppr"].isna().all()
print(
    {
        "rows_2015_2025": int((frame["season"] < FORECAST_SEASON).sum()),
        "holdout_rows": int((frame["season"] == HOLDOUT_SEASON).sum()),
        "forecast_rows": int((frame["season"] == FORECAST_SEASON).sum()),
        "unmatched": int((match_audit["method"] == "unmatched").sum()),
    }
)
source_audit.tail(3)


## Define the model table

All features are available before the target season. Position is a numeric code; the model does not receive player name, team name, season, or the outcome as an input. Unavailable numeric values in the five-season lag block are zero-filled; other missing preseason metadata remains missing.


In [ ]:
BASE_FEATURES = [
    "adp",
    "adp_sd",
    "times_drafted",
    "adp_high",
    "adp_low",
    "age",
    "rookie",
    "years_since_draft",
    "pro_experience",
    "draft_round",
    "draft_pick",
    "height",
    "weight",
]
RECENT_LAG_FEATURES = [
    feature
    for lag in (1, 2)
    for feature in [
        *[f"prev{lag}_{column}" for column in STAT_COLUMNS],
        f"prev{lag}_ppg",
    ]
]
EXTRA_LAG_FEATURES = [
    *[
        f"prev{lag}_{column}"
        for lag in range(3, N_LAGS + 1)
        for column in STAT_COLUMNS
    ],
    *[f"prev{lag}_ppg" for lag in range(3, N_LAGS + 1)],
]
FEATURES = [
    *BASE_FEATURES,
    *RECENT_LAG_FEATURES,
    "changed_team",
    "points_trend",
    "ppg_trend",
    "position_code",
    *EXTRA_LAG_FEATURES,
]


def prepare_matrix(rows: pd.DataFrame) -> np.ndarray:
    work = rows.copy()
    work["position_code"] = work["position"].map(POSITION_CODE)
    return work[FEATURES].to_numpy(dtype=np.float32)


def table_sha256(rows: pd.DataFrame) -> str:
    return sha256_bytes(rows.to_csv(index=False, lineterminator="\n").encode())


assert len(FEATURES) == 102
assert "prev1_fantasy_points_ppr" in FEATURES
assert all(f"prev{lag}_fantasy_points_ppr" in FEATURES for lag in range(1, N_LAGS + 1))
assert not {"season", "player_id", "player_name", "team", "actual_ppr"} & set(FEATURES)

feature_table_hash = table_sha256(frame)
if EXPECTED_FEATURE_TABLE_SHA256 and feature_table_hash != EXPECTED_FEATURE_TABLE_SHA256:
    warnings.warn("The assembled table hash differs from the frozen publication input.")

print({"feature_count": len(FEATURES), "feature_table_sha256": feature_table_hash})


## 2025 holdout

Nori receives 2015–2024 as labeled context and predicts 2025. P10, P50, and P90 come from one predictive distribution; P50 is the point forecast used for MAE.

P50 is requested directly as the median; the same run confirmed that it exactly matches the middle output of quantiles=[0.1, 0.5, 0.9]. The normalized five-lag pipeline scored 51.49 MAE on this holdout.


In [ ]:
checkpoint_path = hf_hub_download(repo_id="Synthefy/Nori", filename="nori.pt")
checkpoint_hash = sha256_file(checkpoint_path)
assert checkpoint_hash == EXPECTED_NORI_6M_SHA256, (
    "The public Nori-6M checkpoint changed; review before comparing results."
)

train_2025 = frame.loc[frame["season"] < HOLDOUT_SEASON].copy()
holdout_2025 = frame.loc[frame["season"] == HOLDOUT_SEASON].copy()
X_train_2025 = prepare_matrix(train_2025)
X_holdout_2025 = prepare_matrix(holdout_2025)
y_train_2025 = train_2025["actual_ppr"].to_numpy(dtype=np.float64)
y_holdout_2025 = holdout_2025["actual_ppr"].to_numpy(dtype=np.float64)

nori_2025 = NoriRegressor(model=MODEL)
nori_2025.fit(X_train_2025, y_train_2025)
p10_2025, p90_2025 = nori_2025.predict(
    X_holdout_2025,
    output_type="quantiles",
    quantiles=[0.1, 0.9],
)
p50_2025 = nori_2025.predict(X_holdout_2025, output_type="median")

mae_2025 = mean_absolute_error(y_holdout_2025, p50_2025)
coverage_2025 = np.mean((y_holdout_2025 >= p10_2025) & (y_holdout_2025 <= p90_2025))
np.testing.assert_allclose(
    mae_2025,
    EXPECTED_2025_MAE,
    rtol=0.0,
    atol=1e-9,
)

pd.DataFrame(
    {
        "metric": ["MAE (lower is better)", "P10–P90 coverage"],
        "2025 holdout": [mae_2025, coverage_2025],
    }
).round(3)


## Forecast and rank 2026

The final fit adds the realized 2025 labels to the context and predicts the outcome-blind 2026 rows. The published list is sorted only by P50 expected points—there is no positional-value adjustment.


In [ ]:
context_2026 = frame.loc[frame["season"] < FORECAST_SEASON].copy()
query_2026 = frame.loc[frame["season"] == FORECAST_SEASON].copy()

nori_2026 = NoriRegressor(model=MODEL)
nori_2026.fit(
    prepare_matrix(context_2026),
    context_2026["actual_ppr"].to_numpy(dtype=np.float64),
)
X_query_2026 = prepare_matrix(query_2026)
p10_2026, p90_2026 = nori_2026.predict(
    X_query_2026,
    output_type="quantiles",
    quantiles=[0.1, 0.9],
)
p50_2026 = nori_2026.predict(X_query_2026, output_type="median")

predictions_2026 = query_2026[
    ["player_id", "player_name", "position", "team", "adp"]
].copy()
predictions_2026["p10"] = p10_2026
predictions_2026["p50"] = p50_2026
predictions_2026["p90"] = p90_2026

board_2026 = (
    predictions_2026.sort_values(
        ["p50", "adp", "player_name"],
        ascending=[False, True, True],
        kind="stable",
    )
    .head(200)
    .reset_index(drop=True)
)
board_2026.insert(0, "rank", np.arange(1, len(board_2026) + 1))
assert board_2026["p50"].is_monotonic_decreasing
assert (board_2026["p10"] <= board_2026["p50"]).all()
assert (board_2026["p50"] <= board_2026["p90"]).all()

board_2026.head(20).round({"adp": 1, "p10": 1, "p50": 1, "p90": 1})


In [ ]:
# Optional export for a draft sheet or website table.
board_2026.to_csv("synthefy_nori_2026_fantasy_top_200.csv", index=False)
print("Wrote synthefy_nori_2026_fantasy_top_200.csv")


## Stafford audit: is last year's production really an input?

Yes. This cell shows the five lag seasons supplied for Matthew Stafford, verifies that his 2025 PPR total is the first lag on the 2026 row, and reports both his prior-season QB finish and Nori's new forecast rank. Games played are inputs too, so the model can distinguish the same point total accumulated over different numbers of games. A low forecast rank can therefore reflect regression toward the mean, age, market information, and older seasons—not omitted 2025 production. The published model does not use injury-report labels; an injury without a missed game is represented only through its effect on production.


In [ ]:
stafford_row = frame.loc[
    (frame["season"] == FORECAST_SEASON)
    & frame["player_name"].str.contains("Stafford", case=False, na=False)
].iloc[0]

stafford_history = pd.DataFrame(
    [
        {
            "season": FORECAST_SEASON - lag,
            "PPR points used": stafford_row[f"prev{lag}_fantasy_points_ppr"],
            "games": stafford_row[f"prev{lag}_games"],
            "points/game": stafford_row[f"prev{lag}_ppg"],
        }
        for lag in range(1, N_LAGS + 1)
    ]
)

qb_2025 = season_stats.loc[
    (season_stats["season"] == HOLDOUT_SEASON)
    & (season_stats["position"] == "QB")
].copy()
qb_2025["prior_rank"] = qb_2025["fantasy_points_ppr"].rank(
    method="min", ascending=False
)
stafford_2025 = qb_2025.loc[
    qb_2025["player_id"] == stafford_row["player_id"]
].iloc[0]
stafford_forecast = predictions_2026.loc[
    predictions_2026["player_id"] == stafford_row["player_id"]
].iloc[0]
qb_forecasts = predictions_2026.loc[predictions_2026["position"] == "QB"].copy()
stafford_qb_rank = int(
    qb_forecasts["p50"].rank(method="min", ascending=False).loc[stafford_forecast.name]
)
stafford_board_row = board_2026.loc[
    board_2026["player_id"] == stafford_row["player_id"]
]
stafford_overall_rank = (
    int(stafford_board_row.iloc[0]["rank"]) if not stafford_board_row.empty else None
)

assert np.isclose(
    stafford_row["prev1_fantasy_points_ppr"],
    stafford_2025["fantasy_points_ppr"],
)
print(
    {
        "2025_full_season_QB_rank": int(stafford_2025["prior_rank"]),
        "2026_Nori_QB_rank": stafford_qb_rank,
        "2026_top_200_rank": stafford_overall_rank,
        "2026_expected_points": round(float(stafford_forecast["p50"]), 1),
    }
)
stafford_history.round(2)


## What this result does—and does not—say

- The 2025 split is chronological: no 2025 outcome appears in its inputs.
- The 2026 table is a preseason forecast, not a live injury or depth-chart feed.
- Fantasy Football Calculator responses are mutable. The notebook prints every response hash and warns when the 2026 source no longer matches the September 2 snapshot; a later rerun can legitimately produce a different board.
- nflverse player data is licensed CC BY 4.0. Fantasy Football Calculator requests attribution for its free API.
- Unavailable numeric lag history is represented as zero; that convention should be reconsidered for players returning after long absences.
